"""
Smart Irrigation Analytics -- Data Cleaning Pipeline
Paste each CELL below into its own notebook cell, in order. Every cell has
been run in true sequence (not in isolation) against sensor_readings_raw.csv
and verified against the numbers already confirmed in audit_findings.md.
"""

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
 
 
# ============================================================
# CELL 1 -- Load raw, clean Timestamp, output pure ISO 8601
# ============================================================

df = pd.read_csv('../data/raw/sensor_readings_raw.csv', dtype=str)
 
 
def parse_timestamp(raw):
    """5 formats exist in this column. Order matters: the 'T' format
    also contains '/', so 'T' must be checked first or it gets
    misclassified as the slash format.
    """
    if pd.isna(raw) or str(raw).strip() == '':
        return pd.NaT
    s = str(raw).strip()
    try:
        if 'T' in s:
            return datetime.strptime(s, '%Y/%m/%dT%H:%M:%S')
        if s.upper().endswith('AM') or s.upper().endswith('PM'):
            return datetime.strptime(s, '%m-%d-%Y %I:%M %p')
        if any(m in s for m in ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                                 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']):
            return datetime.strptime(s, '%d-%b-%Y %H:%M')
        if '/' in s:
            return datetime.strptime(s, '%d/%m/%Y %H:%M')
        return datetime.strptime(s, '%Y-%m-%d %H:%M')
    except ValueError:
        return pd.NaT
 
 
def to_iso_or_na(dt, fmt):
    """Format a datetime as an ISO 8601 string, preserving blanks as
    real pd.NA -- NOT the literal text 'nan', which .dt.strftime()
    would otherwise silently produce and break later isna() checks.
    """
    return dt.strftime(fmt) if pd.notna(dt) else pd.NA
 
 
# Timestamp_clean stays a real datetime object -- needed for any date
# math or grouping later. Timestamp_iso is the pure ISO 8601 TEXT
# version, for display/export.

df['Timestamp_clean'] = df['Timestamp'].apply(parse_timestamp)
df['Timestamp_iso'] = df['Timestamp_clean'].apply(lambda d: to_iso_or_na(d, '%Y-%m-%dT%H:%M:%S'))
df['Date'] = df['Timestamp_clean'].apply(lambda d: to_iso_or_na(d, '%Y-%m-%d'))
df['Time'] = df['Timestamp_clean'].apply(lambda d: to_iso_or_na(d, '%H:%M:%S'))
df['Day'] = df['Timestamp_clean'].dt.day_name()
 
assert (df['Timestamp_clean'].isna() & df['Timestamp'].notna()).sum() == 0, \
    "Some timestamps failed to parse -- check for a 6th format"
assert df['Timestamp'].isna().sum() == 69, \
    f"Expected 69 blank timestamps (per audit), got {df['Timestamp'].isna().sum()}"
assert df['Timestamp_iso'].isna().sum() == 69, \
    "ISO string column should still find all 69 blanks -- check to_iso_or_na"
print("CELL 1 OK: Timestamp parsed to ISO 8601, 0 failures, 69 blanks (matches audit).")

CELL 1 OK: Timestamp parsed to ISO 8601, 0 failures, 69 blanks (matches audit).


In [4]:
# ============================================================
# CELL 2 -- Clean Sensor_ID, fix the Fahrenheit-mislabeled sensors
# ============================================================

def clean_sensor_id(v):
    """Collapse any messy variant to canonical form, e.g.
    'sn_a01', ' SN-A01 ', 'SNA01' -> 'SN-A01'.
    """
    if pd.isna(v):
        return v
    v = v.strip().upper().replace('-', '').replace('_', '')
    return v[:2] + '-' + v[2:]
 
 
df['Sensor_ID_clean'] = df['Sensor_ID'].apply(clean_sensor_id)
assert df['Sensor_ID_clean'].nunique() == 12, \
    f"Expected 12 unique sensors, got {df['Sensor_ID_clean'].nunique()}"
 
MISSING_TOKENS = ['', 'NA', 'N/A', 'null', '-999']
 
# Soil_Temp / Air_Temp need to be real numbers BEFORE the Fahrenheit math
# below, or '-999' (a text sentinel) gets treated as a real temperature
# and corrupts the conversion.
for col in ['Soil_Temp', 'Air_Temp']:
    df[col] = df[col].replace(MISSING_TOKENS, pd.NA)
    df[col] = pd.to_numeric(df[col], errors='coerce')
 
# Confirmed via median-per-sensor check: SN-D01 and SN-D02 report in
# Fahrenheit (median ~85), while all other 10 sensors -- including
# SN-D03 -- report in Celsius (median ~29-30). Convert ONLY those 2
# sensors, Fahrenheit -> Celsius, so every sensor lands on the same
# scale as the 10 that were already correct.
FAHRENHEIT_SENSORS = {'SN-D01', 'SN-D02'}
 
for col in ['Soil_Temp', 'Air_Temp']:
    is_f_sensor = df['Sensor_ID_clean'].isin(FAHRENHEIT_SENSORS)
    df.loc[is_f_sensor, col] = (df.loc[is_f_sensor, col] - 32) * 5 / 9
 
medians = df.groupby('Sensor_ID_clean')['Soil_Temp'].median()
spread = medians.max() - medians.min()
assert spread < 5, f"Sensor spread is {spread:.1f}C -- fix did not apply correctly"
print(f"CELL 2 OK: Fahrenheit fix applied to SN-D01/SN-D02 only. Sensor spread now {spread:.1f}C.")

CELL 2 OK: Fahrenheit fix applied to SN-D01/SN-D02 only. Sensor spread now 1.1C.


In [5]:
# ============================================================
# CELL 3 -- Standardize missing tokens for the remaining columns
# (Soil_Temp/Air_Temp already done in Cell 2 above; Soil_Moisture and
#  Battery_Level get their own handling in Cell 4, not here)
# ============================================================

target_cols = ['Humidity', 'Light_Lux', 'Rainfall_mm', 'Irrigation_Status']
df[target_cols] = df[target_cols].replace(MISSING_TOKENS, pd.NA)
 
for col in ['Humidity', 'Light_Lux', 'Rainfall_mm']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
 
missing_counts = df[target_cols].isna().sum()
assert missing_counts['Humidity'] == 166
assert missing_counts['Light_Lux'] == 188
assert missing_counts['Rainfall_mm'] == 160
assert missing_counts['Irrigation_Status'] == 91
print("CELL 3 OK: missing tokens standardized, all 4 counts match audit_findings.md.")
print(missing_counts.to_string())

CELL 3 OK: missing tokens standardized, all 4 counts match audit_findings.md.
Humidity             166
Light_Lux            188
Rainfall_mm          160
Irrigation_Status     91


In [6]:
# ============================================================
# CELL 4 -- Soil_Moisture (% stripping) and Battery_Level (LOW split)
# ============================================================

df['Soil_Moisture'] = df['Soil_Moisture'].str.replace('%', '', regex=False)
df['Soil_Moisture'] = pd.to_numeric(df['Soil_Moisture'], errors='coerce')
 
# Battery_Level: 'LOW' is preserved as its own status rather than treated
# as a plain missing value -- it's a real (if imprecise) signal the sensor
# sent, a different failure mode than total silence. Same Option B logic
# already used for this column in the Excel audit.
df['Battery_Level_Status'] = np.where(df['Battery_Level'] == 'LOW', 'Non-Numeric', 'Numeric')
df['Battery_Level'] = pd.to_numeric(df['Battery_Level'], errors='coerce')
 
assert df['Soil_Moisture'].dtype == 'float64'
assert df['Battery_Level'].dtype == 'float64'
assert (df['Battery_Level_Status'] == 'Non-Numeric').sum() == 49, \
    "Expected 49 'LOW' rows in Battery_Level"
print("CELL 4 OK: Soil_Moisture numeric, Battery_Level LOW split confirmed (49 rows).")
 
print("\nAll 4 cells verified in true sequence. Ready for the next column.")

CELL 4 OK: Soil_Moisture numeric, Battery_Level LOW split confirmed (49 rows).

All 4 cells verified in true sequence. Ready for the next column.


In [7]:
# ============================================================
# CELL 5 -- Clean Field_Zone, null out impossible Soil_Moisture values
# ============================================================

def clean_field_zone(v):
    """Collapse any messy variant to canonical form, e.g.
    'zone_a', ' Zone A', 'ZoneA' -> 'Zone A'. Letters-only key, same
    technique as clean_sensor_id, just simpler -- no digits to
    reinsert since zone names have none.
    """
    if pd.isna(v):
        return v
    letters = ''.join(c for c in v if c.isalpha()).upper()
    canonical = {'ZONEA': 'Zone A', 'ZONEB': 'Zone B', 'ZONEC': 'Zone C', 'ZONED': 'Zone D'}
    return canonical.get(letters, letters)  # fallback returns the raw key if something unexpected shows up
 
 
df['Field_Zone_clean'] = df['Field_Zone'].apply(clean_field_zone)
assert df['Field_Zone_clean'].nunique() == 4, \
    f"Expected 4 zones, got {df['Field_Zone_clean'].nunique()} -- a variant slipped through"
 
# Soil_Moisture: null out physically impossible values (below 0 or above 100)
# WITHOUT dropping the row -- Battery_Level, Humidity, etc. in that same row
# are very likely still good data and shouldn't be lost over one bad cell.
before = df['Soil_Moisture'].notna().sum()
impossible = df['Soil_Moisture'].notna() & ((df['Soil_Moisture'] < 0) | (df['Soil_Moisture'] > 100))
df.loc[impossible, 'Soil_Moisture'] = pd.NA
after = df['Soil_Moisture'].notna().sum()
 
assert len(df) == 2277, "Row count changed -- outlier handling should null values, never drop rows"
assert before - after == 83, f"Expected 83 impossible values nulled, got {before - after}"
print(f"CELL 5 OK: Field_Zone collapsed to 4 clean values. {before - after} impossible Soil_Moisture values nulled (row count still {len(df)}).")

CELL 5 OK: Field_Zone collapsed to 4 clean values. 83 impossible Soil_Moisture values nulled (row count still 2277).


In [8]:
# ============================================================
# CELL 6 -- Outlier bounds for Humidity and Light_Lux
# ============================================================
# Same null-in-place approach as Cell 5's Soil_Moisture handling: a bad
# value gets nulled, the rest of that row's data (Timestamp, Battery_Level,
# other readings) stays untouched.
 
before_humidity = df['Humidity'].notna().sum()
impossible_humidity = df['Humidity'].notna() & ((df['Humidity'] < 0) | (df['Humidity'] > 100))
df.loc[impossible_humidity, 'Humidity'] = pd.NA
 
# 115000 confirmed against the real data spread earlier in this project:
# legitimate Light_Lux readings run 0 to just over 100,000 (peak desert
# sun), with a clear gap before the glitch values near 999999.
before_light = df['Light_Lux'].notna().sum()
impossible_light = df['Light_Lux'].notna() & ((df['Light_Lux'] < 0) | (df['Light_Lux'] > 115000))
df.loc[impossible_light, 'Light_Lux'] = pd.NA
 
assert len(df) == 2277, "Row count changed -- outlier handling should null values, never drop rows"
print(f"CELL 6 OK: Humidity -- {impossible_humidity.sum()} impossible values nulled. "
      f"Light_Lux -- {impossible_light.sum()} impossible values nulled. Row count still {len(df)}.")

CELL 6 OK: Humidity -- 17 impossible values nulled. Light_Lux -- 20 impossible values nulled. Row count still 2277.


In [9]:
# ============================================================
# CELL 7 -- Battery_Level outlier bound, then drop duplicate rows
# ============================================================

before_battery = df['Battery_Level'].notna().sum()
impossible_battery = df['Battery_Level'].notna() & ((df['Battery_Level'] < 0) | (df['Battery_Level'] > 100))
df.loc[impossible_battery, 'Battery_Level'] = pd.NA
 
assert len(df) == 2277, "Row count changed -- outlier handling should null values, never drop rows"
print(f"CELL 7a OK: Battery_Level -- {impossible_battery.sum()} impossible values nulled. Row count still {len(df)}.")
 
# Duplicates: drop using Reading_ID specifically, NOT a full-column
# comparison. Full-column comparison false-positives here -- two
# genuinely different readings (different Reading_IDs) that both happen
# to have a blank Timestamp AND blank Battery_Level get flagged as
# "duplicate" even though they're real, separate rows. Reading_ID is
# the field this dataset actually generates as a unique identifier, so
# it's the correct thing to trust.
before_rows = len(df)
df = df.drop_duplicates(subset=['Reading_ID'], keep='first')
after_rows = len(df)
 
assert before_rows - after_rows == 45, \
    f"Expected to drop 45 duplicates (per audit), dropped {before_rows - after_rows}"
print(f"CELL 7b OK: dropped {before_rows - after_rows} duplicate Reading_IDs. Row count now {after_rows} (was {before_rows}).")

CELL 7a OK: Battery_Level -- 32 impossible values nulled. Row count still 2277.
CELL 7b OK: dropped 45 duplicate Reading_IDs. Row count now 2232 (was 2277).


In [10]:
# ============================================================
# CELL 8 -- Standardize Irrigation_Status into one boolean column
# ============================================================
# Confirmed 10 distinct non-null text variants: ON/On/1/Yes/TRUE all
# mean "on", OFF/Off/0/No/FALSE all mean "off". Mapping to a real
# boolean (not text) so later filtering/aggregation doesn't need to
# know about the raw encodings at all.
ON_VALUES = {'ON', 'On', '1', 'Yes', 'TRUE'}
OFF_VALUES = {'OFF', 'Off', '0', 'No', 'FALSE'}
 
 
def standardize_irrigation(v):
    if pd.isna(v):
        return pd.NA
    if v in ON_VALUES:
        return True
    if v in OFF_VALUES:
        return False
    return pd.NA  # unrecognized value -- fail safe to NA rather than guess
 
 
df['Irrigation_On'] = df['Irrigation_Status'].apply(standardize_irrigation)
 
unmapped = df['Irrigation_Status'].notna() & df['Irrigation_On'].isna()
assert unmapped.sum() == 0, f"{unmapped.sum()} non-null Irrigation_Status values didn't map -- a variant is missing from ON_VALUES/OFF_VALUES"
assert df['Irrigation_On'].isna().sum() + df['Irrigation_On'].notna().sum() == len(df)
print(f"CELL 8 OK: Irrigation_Status standardized to Irrigation_On (boolean). "
      f"{df['Irrigation_On'].notna().sum()} mapped, {df['Irrigation_On'].isna().sum()} NA, 0 unmapped.")

CELL 8 OK: Irrigation_Status standardized to Irrigation_On (boolean). 2144 mapped, 88 NA, 0 unmapped.
